# ASG Airlines End-to-End Data Engineering Project
## Step 6: Data Transformation & Analytical Field Engineering

---

### 1. Objective
The primary objective of **Step 6: Data Transformation** is to enrich the cleaned datasets from `data/processed/` with analytical features required for flight operational and customer behavior analytics.

**Core Mandates:**
- Load `cleaned_flights.csv`, `cleaned_bookings.csv`, `cleaned_passengers.csv`, and `cleaned_payments.csv`.
- Create route strings (`source → destination`).
- Perform time transformations (extract `departure_hour`, `arrival_hour`, categorize `departure_period`).
- Flag overnight flights where flights cross midnight (`overnight_flag`).
- Extract date components for bookings (`booking_year`, `booking_month`, `booking_day`, `booking_dayofweek`) and age groups for passengers (`age_group`).
- **Scope Boundary:** Do NOT compute final KPIs (e.g. Average Duration, Route Traffic, Delay KPIs) or build Power BI models. Duration calculation is deferred to Step 7.

### 2. Input Data
- `data/processed/cleaned_flights.csv` (1,004 rows)
- `data/processed/cleaned_bookings.csv` (1,000 rows)
- `data/processed/cleaned_passengers.csv` (1,000 rows)
- `data/processed/cleaned_payments.csv` (1,000 rows)

In [ ]:
import os
import pandas as pd
import numpy as np

PROCESSED_DIR = os.path.join("..", "data", "processed")
df_fl = pd.read_csv(os.path.join(PROCESSED_DIR, "cleaned_flights.csv"))
df_bk = pd.read_csv(os.path.join(PROCESSED_DIR, "cleaned_bookings.csv"))
df_pass = pd.read_csv(os.path.join(PROCESSED_DIR, "cleaned_passengers.csv"))
df_pay = pd.read_csv(os.path.join(PROCESSED_DIR, "cleaned_payments.csv"))

print(f"Loaded Cleaned Datasets:")
print(f"  - Flights:    {df_fl.shape}")
print(f"  - Bookings:   {df_bk.shape}")
print(f"  - Passengers: {df_pass.shape}")
print(f"  - Payments:   {df_pay.shape}")

### 3. Transformation Strategy
We implement deterministic, modular transformations in Python ensuring strict type safety and zero row loss.

### 4. Data Type Standardization
Ensuring IDs are strings, timestamps are datetime types, and numerical metrics are floats/integers.

In [ ]:
df_fl["departure_time"] = pd.to_datetime(df_fl["departure_time"])
df_fl["arrival_time"] = pd.to_datetime(df_fl["arrival_time"])
print("Flights Data Types:\n", df_fl.dtypes)

### 5. Route Creation
Constructing standard flight route strings in format `source → destination` (e.g. `DEL → BOM`, `CCU → MAA`).

In [ ]:
df_fl["route"] = df_fl["source"].astype(str) + " → " + df_fl["destination"].astype(str)
print("Unique Routes Generated:", df_fl["route"].nunique())
display(df_fl[["flight_id", "source", "destination", "route"]].head(5))

### 6. Time Transformation & Hour Extraction
Extracting `departure_hour` and `arrival_hour` from timestamp fields.

In [ ]:
df_fl["departure_hour"] = df_fl["departure_time"].dt.hour
df_fl["arrival_hour"] = df_fl["arrival_time"].dt.hour
display(df_fl[["flight_id", "departure_time", "departure_hour", "arrival_time", "arrival_hour"]].head(5))

### 7. Departure Period Categorization
Categorizing departure hour into standardized 6-hour operational time windows:
- **Night:** `00:00 - 05:59` (Hours 0-5)
- **Morning:** `06:00 - 11:59` (Hours 6-11)
- **Afternoon:** `12:00 - 17:59` (Hours 12-17)
- **Evening:** `18:00 - 23:59` (Hours 18-23)

In [ ]:
def get_departure_period(hour):
    if 0 <= hour < 6:
        return "Night"
    elif 6 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 18:
        return "Afternoon"
    else:
        return "Evening"

df_fl["departure_period"] = df_fl["departure_hour"].apply(get_departure_period)
print("Departure Period Distribution:\n", df_fl["departure_period"].value_counts())

### 8. Overnight Flight Flag
Flagging flights that span across calendar midnight (`arrival_date > departure_date` or `arrival_hour < departure_hour`).

In [ ]:
is_next_day_date = (df_fl["arrival_time"].dt.date > df_fl["departure_time"].dt.date)
is_next_day_hour = (df_fl["arrival_hour"] < df_fl["departure_hour"])
df_fl["overnight_flag"] = (is_next_day_date | is_next_day_hour).astype(int)

print("Overnight Flights Flagged:", df_fl["overnight_flag"].sum())
display(df_fl[df_fl["overnight_flag"] == 1][["flight_id", "departure_time", "arrival_time", "overnight_flag"]].head(5))

### 9. Relationship Preservation
Verifying that key identifiers (`flight_id`, `passenger_id`, `booking_id`) remain 100% consistent across all datasets.

In [ ]:
print("Bookings -> Flights FK match:", df_bk["flight_id"].isin(df_fl["flight_id"]).all())
print("Bookings -> Passengers FK match:", df_bk["passenger_id"].isin(df_pass["passenger_id"]).all())
print("Payments -> Bookings FK match:", df_pay["booking_id"].isin(df_bk["booking_id"]).all())

### 10. Output Datasets
Saved datasets in `data/processed/`:
- `transformed_flights.csv`
- `transformed_bookings.csv`
- `transformed_passengers.csv`
- `transformed_payments.csv`

### 11. Transformation Summary Table

| Column | Transformation | Reason |
|---|---|---|
| `route` | `source + " → " + destination` | Enable route-level flight operations analytics. |
| `departure_hour` | `departure_time.dt.hour` | Analyze peak departure times. |
| `arrival_hour` | `arrival_time.dt.hour` | Analyze airport arrival schedules. |
| `departure_period` | 6-hr window mapping (`Night`, `Morning`, `Afternoon`, `Evening`) | Segment flight schedules into operational periods. |
| `overnight_flag` | Boolean flag (1 if arrival date > departure date) | Identify cross-midnight flights for crew & scheduling analysis. |
| `booking_year/month/day/dayofweek` | Date component extraction from `booking_date` | Enable time-series booking trend analysis. |
| `age_group` | Categorical mapping (`Child`, `Youth`, `Adult`, `Senior`) | Segment passengers for demographic analytics. |

### 12. Assumptions
1. **Departure Periods:** Standard 6-hour windows reflect airline operational shifts.
2. **Overnight Flights:** Flights where arrival hour is less than departure hour or arrival date is after departure date span across calendar days.
3. **Preservation:** No final duration or KPI calculations were performed, preserving boundaries for Step 7.

### 13. Conclusion
**Step 6: Data Transformation** completed cleanly. All datasets have been enriched with analytical fields and saved to `data/processed/transformed_*.csv`.